# 2D Inverse FFT Interactive Demo

任意の画像が正弦波の重ね合わせであることをインタラクティブに学ぶデモです。

**操作方法:**
1. セルをすべて実行する
2. 振幅スペクトル（上中・下中パネル）の上でマウスをドラッグする
3. 通過した周波数成分が逐次 IFFT に加算され、下左パネルに再合成画像が現れる
4. 下右パネルに最後に追加したグレーティング（正弦波成分）が表示される
5. Reset ボタンでマスクをリセットできる

---
Python port of: Sasaki & Ohzawa (2009), Osaka University  
Original license: BSD

In [ ]:
# Google Colab 環境セットアップ
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q ipympl
    from google.colab import output
    output.enable_custom_widget_manager()

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Button
from PIL import Image
import urllib.request
import os

In [ ]:
# -----------------------------------------------------------------
# 画像の準備
# -----------------------------------------------------------------
# Colab では下記のいずれかで画像を用意してください:
#   A) 同梱サンプル画像URL から自動ダウンロード（デフォルト）
#   B) Google Drive をマウントして path を指定
#   C) ローカルからアップロード
#
# ローカル実行 (Jupyter等) では images/ フォルダのパスを直接指定できます

# --- サンプル画像をGitHubから取得する場合 (Colab用) ---
GITHUB_RAW = "https://raw.githubusercontent.com/AquilaITOH/2DIFFT_demo/main/MATLAB/InverseFFT2D/images/"
SAMPLE_IMAGES = [
    "TaiyounoTou128x128t.png",
    "lena_std128x128.png",
    "Albert_Einstein_Nobel_s.png",
]

def fetch_image_if_needed(filename):
    """ローカルになければGitHubからダウンロード"""
    if not os.path.exists(filename):
        url = GITHUB_RAW + os.path.basename(filename)
        try:
            urllib.request.urlretrieve(url, filename)
        except Exception:
            return False
    return True

# --- 使用する画像ファイルパスをここで設定 ---
IMAGE_PATH = "TaiyounoTou128x128t.png"  # ← 変更可

# ローカルの images/ フォルダにある場合はそちらを優先
if os.path.exists(os.path.join("images", IMAGE_PATH)):
    IMAGE_PATH = os.path.join("images", IMAGE_PATH)
else:
    fetch_image_if_needed(IMAGE_PATH)

print(f"使用画像: {IMAGE_PATH}")

In [ ]:
# -----------------------------------------------------------------
# コア関数
# -----------------------------------------------------------------

def get_luminance_image(filename):
    """画像を読み込みグレースケール輝度に変換（MATLABのGetLuminanceImage相当）"""
    img = Image.open(filename)
    X = np.array(img, dtype=float)
    if X.ndim == 3:
        L = 0.30 * X[:, :, 0] + 0.59 * X[:, :, 1] + 0.11 * X[:, :, 2]
    else:
        L = X
    return L - 127.5


def myff2(X, m=None, n=None):
    """2D FFT + fftshift、周波数軸を返す（MATLABのMyff2相当）"""
    if m is None:
        m, n = X.shape
    Y = np.fft.fftshift(np.fft.fft2(X, s=(m, n)))
    f0 = np.floor(np.array([m, n]) / 2).astype(int) + 1  # 1-indexed center
    # MATLABの fy = ((m:-1:1) - f0(1) + 1) / m  に対応
    fy = (np.arange(m, 0, -1) - f0[0] + 1) / m   # 上→下が高周波→低周波
    fx = (np.arange(1, n + 1) - f0[1]) / n
    return Y, fx, fy


def draw_line_on_mask(mask, xi_new, yi_new, xi_old, yi_old):
    """2点間を補間してマスクに線を描く（MATLABのUpdateMask内の補間ロジック相当）"""
    if xi_old is None:
        mask[yi_new, xi_new] = 1
        return mask

    xi, yi = xi_new, yi_new
    xo, yo = xi_old, yi_old

    if xi == xo:
        yy = np.arange(min(yi, yo), max(yi, yo) + 1)
        xx = np.full_like(yy, xi)
    else:
        slope = (yi - yo) / (xi - xo)
        if abs(slope) < 1:
            xx = np.arange(min(xi, xo), max(xi, xo) + 1)
            yy = np.round(slope * (xx - xi) + yi).astype(int)
        else:
            yy = np.arange(min(yi, yo), max(yi, yo) + 1)
            xx = np.round((1 / slope) * (yy - yi) + xi).astype(int)

    # 範囲クリップ
    rows, cols = mask.shape
    valid = (xx >= 0) & (xx < cols) & (yy >= 0) & (yy < rows)
    mask[yy[valid], xx[valid]] = 1
    return mask

In [ ]:
# -----------------------------------------------------------------
# デモ本体クラス
# -----------------------------------------------------------------

class IFFTDemo:
    MIN_AMP = 1e-10

    def __init__(self, filename):
        self.filename = filename
        self._load_and_init()
        self._build_figure()
        self._connect_events()
        self._update_ifft()

    # ----------------------------------------------------------
    # 初期化
    # ----------------------------------------------------------
    def _load_and_init(self):
        self.L = get_luminance_image(self.filename)
        fft_pts = 2 ** np.ceil(np.log2(self.L.shape)).astype(int)
        self.FFTedL, self.fx, self.fy = myff2(self.L, fft_pts[0], fft_pts[1])
        self.mask = np.zeros(self.FFTedL.shape)

        amp = np.abs(self.FFTedL)
        amp = np.where(amp < self.MIN_AMP, self.MIN_AMP, amp)
        self.amp = np.log10(amp)

        self._dragging = False
        self._old_xi = None
        self._old_yi = None
        self._picking_ax = None

    # ----------------------------------------------------------
    # 図の構築
    # ----------------------------------------------------------
    def _build_figure(self):
        self.fig = plt.figure(figsize=(12, 8))
        self.fig.canvas.header_visible = False

        gs = gridspec.GridSpec(2, 3, figure=self.fig,
                               hspace=0.4, wspace=0.35)

        self.ax_orig   = self.fig.add_subplot(gs[0, 0])
        self.ax_amp    = self.fig.add_subplot(gs[0, 1])
        self.ax_ifft   = self.fig.add_subplot(gs[1, 0])
        self.ax_picker = self.fig.add_subplot(gs[1, 1])
        self.ax_grat   = self.fig.add_subplot(gs[1, 2])

        cmap = 'gray'
        # extent = [left, right, bottom, top] in data coords
        ext = [self.fx[0], self.fx[-1], self.fy[-1], self.fy[0]]

        # 元画像
        self.ax_orig.imshow(self.L, cmap=cmap, origin='upper', aspect='equal')
        self.ax_orig.set_title('original image')
        self.ax_orig.set_xlabel('x')
        self.ax_orig.set_ylabel('y')

        # 振幅スペクトル（クリック受付）
        self.im_amp = self.ax_amp.imshow(
            self.amp, cmap=cmap, origin='upper', extent=ext, aspect='equal')
        self.ax_amp.set_title('amplitude spectrum')
        self.ax_amp.set_xlabel('fx (cyc/pix)')
        self.ax_amp.set_ylabel('fy (cyc/pix)')

        # IFFTed image
        blank = np.zeros(self.L.shape)
        self.im_ifft = self.ax_ifft.imshow(
            blank, cmap=cmap, origin='upper', aspect='equal')
        self.ax_ifft.set_title('IFFTed image')
        self.ax_ifft.set_xlabel('x')
        self.ax_ifft.set_ylabel('y')

        # Picker（選択済みスペクトル）
        self.im_picker = self.ax_picker.imshow(
            np.zeros_like(self.amp), cmap=cmap, origin='upper',
            extent=ext, aspect='equal')
        self.ax_picker.set_title('unmasked amplitude spectrum')
        self.ax_picker.set_xlabel('fx (cyc/pix)')
        self.ax_picker.set_ylabel('fy (cyc/pix)')

        # 最後のグレーティング
        self.im_grat = self.ax_grat.imshow(
            blank, cmap=cmap, origin='upper', aspect='equal')
        self.ax_grat.set_title('most recent grating added')
        self.ax_grat.set_xlabel('x')
        self.ax_grat.set_ylabel('y')

        # Reset ボタン
        ax_btn = self.fig.add_axes([0.78, 0.92, 0.1, 0.04])
        self.btn_reset = Button(ax_btn, 'Reset')
        self.btn_reset.on_clicked(self._on_reset)

        self.fig.canvas.draw()

    # ----------------------------------------------------------
    # イベント接続
    # ----------------------------------------------------------
    def _connect_events(self):
        c = self.fig.canvas
        c.mpl_connect('button_press_event',   self._on_press)
        c.mpl_connect('motion_notify_event',  self._on_motion)
        c.mpl_connect('button_release_event', self._on_release)

    def _is_picker_ax(self, ax):
        return ax in (self.ax_amp, self.ax_picker)

    # ----------------------------------------------------------
    # マウスイベントハンドラ
    # ----------------------------------------------------------
    def _on_press(self, event):
        if event.inaxes is None or not self._is_picker_ax(event.inaxes):
            return
        self._dragging = True
        self._picking_ax = event.inaxes
        self._old_xi = None
        self._old_yi = None
        self._process_event(event)

    def _on_motion(self, event):
        if not self._dragging:
            return
        if event.inaxes is None or not self._is_picker_ax(event.inaxes):
            self._dragging = False
            return
        self._process_event(event)

    def _on_release(self, event):
        self._dragging = False
        self._picking_ax = None
        self._old_xi = None
        self._old_yi = None

    def _on_reset(self, event):
        self.mask[:] = 0
        self._old_xi = None
        self._old_yi = None
        self._update_ifft()

    # ----------------------------------------------------------
    # マスク更新
    # ----------------------------------------------------------
    def _process_event(self, event):
        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return

        xi = int(np.argmin(np.abs(self.fx - x)))
        yi = int(np.argmin(np.abs(self.fy - y)))

        draw_line_on_mask(self.mask, xi, yi, self._old_xi, self._old_yi)
        self._old_xi = xi
        self._old_yi = yi
        self._update_ifft()

    # ----------------------------------------------------------
    # IFFT 計算 & 表示更新
    # ----------------------------------------------------------
    def _update_ifft(self):
        # 再合成画像
        A = np.real(np.fft.ifft2(np.fft.ifftshift(self.mask * self.FFTedL)))
        A = A[:self.L.shape[0], :self.L.shape[1]]

        # 選択済みスペクトル
        picked_amp = self.mask * self.amp

        # 最後のグレーティング
        G = np.zeros_like(self.mask)
        if self._old_xi is not None:
            G[self._old_yi, self._old_xi] = 1
            G = np.real(np.fft.ifft2(np.fft.ifftshift(G * self.FFTedL)))
            G = G[:self.L.shape[0], :self.L.shape[1]]

        self.im_ifft.set_data(A)
        self.im_ifft.set_clim(vmin=A.min(), vmax=A.max())

        self.im_picker.set_data(picked_amp)
        self.im_picker.set_clim(vmin=picked_amp.min(), vmax=picked_amp.max())

        if self._old_xi is not None:
            self.im_grat.set_data(G)
            self.im_grat.set_clim(vmin=G.min(), vmax=G.max())

        self.fig.canvas.draw_idle()

In [ ]:
# -----------------------------------------------------------------
# 実行
# -----------------------------------------------------------------
plt.close('all')
demo = IFFTDemo(IMAGE_PATH)